In [1]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'crunchbase_texts_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [2]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    # .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

8696

In [3]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/cb/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

In [4]:
labelled_df = (
    pd.concat(dfs, ignore_index=True)
    # Key step: taking only data that's robustly relevant
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
)

In [5]:
labelled_text_df = (
    text_df
    .merge(labelled_df, on='id', how='left')
    .set_index("id")
)

In [6]:
# Double check specific topics
extra_keywords = {
    "ai2": ["artificial intelligence", "data science", "machine learning", "deep learning", "chatbot", "natural language processing", "computer vision", "convolutional neural network", "recurrent neural network", "reinforcement learning", "predictive model", "predictive analytics"],
    "ar_vr": ["virtual reality", "augmented reality", "mixed reality"],
    "social_media": ["social media"],
    "robotics": ["robot"],
    "parenting2": ["home learning environment", "home learning", "parenting approach", "parenting style", "home learning",
    "parenting style",
    "single parent",
    "parenting skill",
    "parenting education",
    "parenting program",
    "parenting intervention",
    "parenting support",
    "parenting practice",
    "parenting behavior",
    "parenting knowledge",
    "parenting attitude",
    "parenting guidance",
    "parenting stress",
    "parent skill",
    "parent education",
    "parent program",
    "parent intervention",
    "parent support",
    "parent practice",
    "parent behavior",
    "parent knowledge",
    "parent attitude",
    "parent guidance",
    "parent stress"],
    "wearables": ["wearable", "internet of things", " iot "],
    "mobile": ["smartphone", 'ipad', 'iphone', 'android', 'phone application'],
    "infancy": ["infant", "newborn", "neonate"],
    "protection": ["child protection", "safeguarding"],
    "communication": ["language development", "speech development"],
    "cognitive": ["cognitive development"],
    "send": ["autism", "adhd", "learning disability", "special educational needs"],
    "mental_health": [" mental health "],
    "rct": ["randomised control trial", "randomized control trial"],
    "social_services": ["social service"],
    "mobile": ['mobile phone', 'smartphone', 'android', 'iphone'],
}
extra_keywords_patents = {
    "preschool": ['preschool']
}


In [7]:
for topic in extra_keywords:
    keywords = extra_keywords[topic]
    keywords = [word.lower() for word in keywords]
    hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
    hits_ids = hits_df.id.to_list()
    labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [136]:
# for topic in extra_keywords_patents:
#     keywords = extra_keywords_patents[topic]
#     keywords = [word.lower() for word in keywords]
#     hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
#     hits_ids = hits_df.query("source == 'patents'").id.to_list()
#     labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [8]:
text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    # .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_text_df.reset_index()[['id', 'topics']], on="id", how="left")
    .assign(topics = lambda df: df.topics.apply(lambda x: ", ".join(x) if type(x) == list else x))
    .drop(columns=["Unnamed: 0", "predictions"])
)

In [9]:
# Papers with no labels
n_with_topics = (text_labelled_df.topics.isnull() == False).sum()
n_without_topics = text_labelled_df.topics.isnull().sum()
n_with_topics / len(text_labelled_df), n_without_topics / len(text_labelled_df)

(0.5765869365225391, 0.4234130634774609)

In [10]:
print(n_with_topics)

5014


In [11]:
text_labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_crunchbase_filtered.csv', index=False)

In [12]:
text_labelled_df.sample(10)

,id,text,topics
6481,4e5d0fe6-1575-4270-9c5f-693b3e518f02,Cay Galgon Life House is a prepare pregnant wo...,infancy
4288,992b927a-efae-4cba-b0e6-87dfca36edfa,Domaomao is a children's snack brand that prov...,NaN
2572,ff01fd31-cd00-46fe-ac9f-6153f4bff013,Detmir is an online store that offers various ...,NaN
6100,8034df7d-c556-498a-8fd2-e732a62913b6,Idas is a manufacturer and supplier of mattres...,NaN
3907,74dcb663-fa34-4b4c-911a-7f6d84d8d3d3,"RIE is an international, non-profit organizati...",infancy
1511,dcb860aa-5fd7-76dc-abf5-0bec000fd746,KindyNow is a last minute childcare booking en...,NaN
4307,816e8e9e-6d1b-4b56-a255-40a56fdfd556,Oregon Pediatrics has experienced Pediatrician...,infancy
3654,7de6c4de-9b83-4a55-925b-0c5b339afa31,Gunjan Apps Studios and Solutions LLP creates ...,"mobile, arts, games"
4989,33f4a5d9-c9b5-465d-bc0a-d84d3043b524,Music Garden provides childhood music educatio...,"infancy, arts"
1982,6d7f3944-63c6-4af2-b2ad-dc6a44c62f98,“TooCuteForMe” is a USA based Online shopping ...,NaN


In [13]:
len(text_labelled_df)

8696

In [19]:
# import pandas as pd
# gtr_metadata_df = pd.read_csv(ENRICHED_DATA_DIR / 'crunchbase_texts.csv')

In [24]:
# (
#     gtr_metadata_df
#     .merge(text_labelled_df[['id', 'topics']], on='id', how='inner')
# ).to_csv(ENRICHED_DATA_DIR / 'gtr_texts_relevant_metadata.csv', index=False)

## Extra checks